# Generate activity scores for the scRNA-seq benchmarks

This notebook generates the activity-score parquet files used by `mwu_delongs.ipynb`. It runs:

1. **z-aggregate** across the prior-network and weighting configurations used in the paper, and
2. **VIPER, ULM, and z-score** for the method comparison.

Datasets are discovered dynamically from `scRNASeq/*.h5ad`. The optional download section can retrieve selected missing datasets from Zenodo.


## Local project layout

The notebook can be launched from the repository root or from `reproduce/Reproduce scRNASeq Results/`.

- Input datasets: `scRNASeq/<dataset>.h5ad`
- Prior networks: repository-level `data/*.tsv`
- Generated scores: `scores/<dataset>/*.parquet`
- Local z-aggregate implementation: repository-level `src/`

The filenames written here match the patterns expected by the MWU and DeLong analysis scripts.


## Setup

Run this cell before the download, discovery, and scoring sections. It resolves project paths, creates the local input and output directories, and loads the repository implementation.

In [13]:
from pathlib import Path
import sys
from urllib.request import urlretrieve

import decoupler as dc
import pandas as pd
import scanpy as sc
from tqdm.auto import tqdm

current_dir = Path.cwd().resolve()
analysis_dir = current_dir / "reproduce" / "Reproduce scRNASeq Results"
analysis_dir = analysis_dir if analysis_dir.is_dir() else current_dir
repository_root = analysis_dir.parents[1]

for path in (repository_root, analysis_dir / "scripts"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from utility_functions import (
    WeightType,
    compute_network_weights,
    preprocess_adata,
    read_adata_file,
    read_prior_network_file,
)
from src.z_aggregate import run_z_aggregate

dataset_directory = analysis_dir / "scRNASeq"
scores_directory = analysis_dir / "scores"
prior_directory = repository_root / "data"

dataset_directory.mkdir(exist_ok=True)
scores_directory.mkdir(exist_ok=True)


Repository root: /Users/kisanthapa/Downloads/z-aggregate
Dataset directory: /Users/kisanthapa/Downloads/z-aggregate/reproduce/Reproduce scRNASeq Results/scRNASeq
Scores directory: /Users/kisanthapa/Downloads/z-aggregate/reproduce/Reproduce scRNASeq Results/scores


## Optional dataset download

Choose the datasets to download below. Existing files in scRNASeq are skipped; set download_missing_datasets to False to skip this step.

In [ ]:
datasets_to_download = (
    "FrangiehIzar2021_RNA",
    "GasperiniShendure2019_lowMOI",
    "NadigOConner2024_hepg2",
    "NadigOConner2024_jurkat",
    "NormanWeissman2019_filtered",
    "PapalexiSatija2021_eccite_RNA",
    "ReplogleWeissman2022_K562_essential",
    "ReplogleWeissman2022_K562_gwps",
    "ReplogleWeissman2022_rpe1",
    "SunshineHein2023",
    "TianKampmann2021_CRISPRa",
    "TianKampmann2021_CRISPRi",
    "WesselsSatija2023",
)
download_missing_datasets = True

if download_missing_datasets:
    for dataset_name in datasets_to_download:
        dataset_path = dataset_directory / f"{dataset_name}.h5ad"
        if not dataset_path.exists():
            print(f"Downloading {dataset_name}")
            urlretrieve(
                f"https://zenodo.org/records/13350497/files/{dataset_name}.h5ad?download=1",
                dataset_path,
            )

## Discover datasets

Every .h5ad file in scRNASeq is treated as one dataset. Discovery runs after the optional download step, so manually added files are included automatically.


In [12]:
dataset_paths = sorted(dataset_directory.glob("*.h5ad"))

if not dataset_paths:
    raise FileNotFoundError(f"No .h5ad files found in {dataset_directory}")

pd.DataFrame({"dataset": [path.stem for path in dataset_paths], "path": dataset_paths})

,dataset,path
0,PapalexiSatija2021_eccite_RNA,/Users/kisanthapa/Downloads/z-aggregate/reprod...
1,TianKampmann2021_CRISPRa,/Users/kisanthapa/Downloads/z-aggregate/reprod...
2,TianKampmann2021_CRISPRi,/Users/kisanthapa/Downloads/z-aggregate/reprod...


## Analysis configuration

The configuration below reproduces the score combinations consumed by the downstream comparisons:

- z-aggregate with four prior networks using uniform weights,
- z-aggregate with four weighting strategies using the CausalPath prior, and
- VIPER, ULM, and z-score using the CausalPath prior with uniform weights.

`overwrite = False` keeps existing parquet files and computes only missing outputs.


In [3]:
min_targets = 5
decoupler_batch_size = 10_000
overwrite = False

prior_paths = {
    "causalpath": prior_directory / "causalpath.tsv",
    "collectri": prior_directory / "collectri.tsv",
    "dorothea": prior_directory / "dorothea.tsv",
    "ensemble": prior_directory / "ensemble.tsv",
}

weight_types = {
    "UNIFORM": WeightType.UNIFORM,
    "CORRELATION": WeightType.CORRELATION,
    "SPECIFICITY": WeightType.SPECIFICITY,
    "NONZERORATE": WeightType.NON_ZERO_RATIO,
}

z_aggregate_configurations = [
    (prior_name, "UNIFORM") for prior_name in prior_paths
] + [
    ("causalpath", weight_name)
    for weight_name in ("CORRELATION", "SPECIFICITY", "NONZERORATE")
]

comparison_methods = ["viper", "ulm", "zscore"]

missing_priors = [str(path) for path in prior_paths.values() if not path.is_file()]
if missing_priors:
    raise FileNotFoundError(f"Missing prior-network files: {missing_priors}")

z_aggregate_configurations


[('causalpath', 'UNIFORM'),
 ('collectri', 'UNIFORM'),
 ('dorothea', 'UNIFORM'),
 ('ensemble', 'UNIFORM'),
 ('causalpath', 'CORRELATION'),
 ('causalpath', 'SPECIFICITY'),
 ('causalpath', 'NONZERORATE')]

## Run z-aggregate with shared utility functions

This section uses the shared helpers in scripts/utility_functions.py for data loading, preprocessing, prior parsing, and network weighting. The z-aggregate score calculation remains in src/z_aggregate.py.

- `utility_functions.read_adata_file` loads each AnnData file.
- `utility_functions.preprocess_adata` performs the shared adaptive filtering, normalization, and log transformation.
- `utility_functions.read_prior_network_file` parses each local prior network.
- `utility_functions.compute_network_weights` applies the selected weighting strategy.
- `src.z_aggregate.run_z_aggregate` computes TF activity scores and p-values.

Only the activity scores are saved because those are the inputs required by the MWU and DeLong analyses. Output files follow:

`<dataset>_z-aggregate_<prior>_<WEIGHT>.parquet`


In [ ]:
def load_preprocessed_dataset(dataset_path: Path):
    return preprocess_adata(read_adata_file(str(dataset_path)), do_scale=False)


def prepare_network(
    adata,
    prior_network: pd.DataFrame,
    weight_type: WeightType,
) -> pd.DataFrame:
    network = prior_network[prior_network["target"].isin(adata.var_names)].copy()
    network = network[
        network.groupby("source")["target"].transform("nunique") >= min_targets
    ]

    return compute_network_weights(
        adata=adata,
        prior_network=network,
        weight_type=weight_type,
    )


prior_networks = {
    name: read_prior_network_file(str(path)) for name, path in prior_paths.items()
}


{'causalpath': 12071, 'collectri': 42871, 'dorothea': 32286, 'ensemble': 75341}

In [5]:
def run_z_aggregate_for_dataset(dataset_path: Path) -> list[dict]:
    dataset_name = dataset_path.stem
    output_directory = scores_directory / dataset_name
    output_directory.mkdir(parents=True, exist_ok=True)

    pending = [
        (prior_name, weight_name)
        for prior_name, weight_name in z_aggregate_configurations
        if overwrite
        or not (output_directory / f"{dataset_name}_z-aggregate_{prior_name}_{weight_name}.parquet").exists()
    ]

    if not pending:
        print(f"Skipping z-aggregate for {dataset_name}: all outputs exist")
        return []

    print(f"Loading and preprocessing {dataset_name}")
    adata = load_preprocessed_dataset(dataset_path)
    rows = []

    for prior_name, weight_name in pending:
        output_path = output_directory / f"{dataset_name}_z-aggregate_{prior_name}_{weight_name}.parquet"
        network = prepare_network(
            adata=adata,
            prior_network=prior_networks[prior_name],
            weight_type=weight_types[weight_name],
        )

        scores, _ = run_z_aggregate(
            adata=adata,
            priors=network,
            min_targets=min_targets,
        )
        scores = scores.sort_index(axis=1)
        scores.to_parquet(output_path, engine="pyarrow")

        rows.append(
            {
                "dataset": dataset_name,
                "method": "z-aggregate",
                "prior": prior_name,
                "weight": weight_name,
                "cells": scores.shape[0],
                "regulators": scores.shape[1],
                "output": str(output_path),
            }
        )
    return rows


In [6]:
z_aggregate_summary = []

for dataset_path in tqdm(dataset_paths, desc="Datasets: z-aggregate"):
    z_aggregate_summary.extend(run_z_aggregate_for_dataset(dataset_path))

pd.DataFrame(z_aggregate_summary)


Datasets: z-aggregate:   0%|          | 0/2 [00:00<?, ?it/s]

2026-06-20 21:11:05 | [INFO] Reading expression data from: /Users/kisanthapa/Downloads/z-aggregate/reproduce/Reproduce scRNASeq Results/scRNASeq/PapalexiSatija2021_eccite_RNA.h5ad


Loading and preprocessing PapalexiSatija2021_eccite_RNA


2026-06-20 21:11:07 | [INFO]    Loaded data shape: 20729 cells x 18649 genes
2026-06-20 21:11:07 | [INFO] Starting Preprocessing...
2026-06-20 21:11:08 | [INFO]    Filtered cells (min_genes=1000): Removed 0 cells.
2026-06-20 21:11:08 | [INFO]    Filtered genes (min_cells=10): Removed 915 genes.
2026-06-20 21:11:08 | [INFO]    Mitochondrial filter (<20.0%): Removed 0 cells.
2026-06-20 21:11:09 | [INFO] Preprocessing complete. Final shape: 20729 cells x 17734 genes
2026-06-20 21:11:09 | [INFO] Computing weights using strategy: Uniform
2026-06-20 21:11:09 | [INFO]    Uniform weights: using interaction as weight (no overlap filtering).
2026-06-20 21:11:09 | [INFO]    Weights computed successfully.


  z-aggregate: prior=causalpath, weight=UNIFORM


...

  z-aggregate: prior=causalpath, weight=SPECIFICITY


2026-06-20 21:14:51 | [INFO] Computing weights using strategy: NonzeroRate
2026-06-20 21:14:51 | [INFO]    Network Overlap: 7484/7484 edges (100.00%) target genes present in dataset.
2026-06-20 21:14:51 | [INFO]    Calculating nonzero rate weights...


  z-aggregate: prior=causalpath, weight=NONZERORATE


2026-06-20 21:14:51 | [INFO]    Weights computed successfully.


,dataset,method,prior,weight,cells,regulators,output
0,PapalexiSatija2021_eccite_RNA,z-aggregate,causalpath,UNIFORM,20729,424,/Users/kisanthapa/Downloads/z-aggregate/reprod...
1,PapalexiSatija2021_eccite_RNA,z-aggregate,collectri,UNIFORM,20729,658,/Users/kisanthapa/Downloads/z-aggregate/reprod...
2,PapalexiSatija2021_eccite_RNA,z-aggregate,dorothea,UNIFORM,20729,290,/Users/kisanthapa/Downloads/z-aggregate/reprod...
3,PapalexiSatija2021_eccite_RNA,z-aggregate,ensemble,UNIFORM,20729,898,/Users/kisanthapa/Downloads/z-aggregate/reprod...
4,PapalexiSatija2021_eccite_RNA,z-aggregate,causalpath,CORRELATION,20729,424,/Users/kisanthapa/Downloads/z-aggregate/reprod...
5,PapalexiSatija2021_eccite_RNA,z-aggregate,causalpath,SPECIFICITY,20729,424,/Users/kisanthapa/Downloads/z-aggregate/reprod...
6,PapalexiSatija2021_eccite_RNA,z-aggregate,causalpath,NONZERORATE,20729,424,/Users/kisanthapa/Downloads/z-aggregate/reprod...
7,TianKampmann2021_CRISPRa,z-aggregate,causalpath,UNIFORM,19155,444,/Users/kisanthapa/Downloads/z-aggregate/reprod...
8,TianKampmann2021_CRISPRa,z-aggregate,collectri,UNIFORM,19155,707,/Users/kisanthapa/Downloads/z-aggregate/reprod...
9,TianKampmann2021_CRISPRa,z-aggregate,dorothea,UNIFORM,19155,290,/Users/kisanthapa/Downloads/z-aggregate/reprod...


## Run VIPER, ULM, and z-score with Decoupler

For the method comparison, each dataset is preprocessed and then scaled before running the three Decoupler methods together. The signed CausalPath network uses uniform weights. Outputs follow:

`<dataset>_<method>_causalpath_UNIFORM.parquet`

These names match the method-comparison loader in `mwu_delongs_scripts/Methods_MWU-Delongs.py`.


In [7]:
def run_comparison_methods_for_dataset(dataset_path: Path) -> list[dict]:
    dataset_name = dataset_path.stem
    output_directory = scores_directory / dataset_name
    output_directory.mkdir(parents=True, exist_ok=True)

    output_paths = {
        method: output_directory
        / f"{dataset_name}_{method}_causalpath_UNIFORM.parquet"
        for method in comparison_methods
    }
    pending_methods = [
        method
        for method, output_path in output_paths.items()
        if overwrite or not output_path.exists()
    ]

    if not pending_methods:
        print(f"Skipping comparison methods for {dataset_name}: all outputs exist")
        return []

    print(f"Loading and preprocessing {dataset_name}")
    adata = load_preprocessed_dataset(dataset_path)
    network = prepare_network(
        adata=adata,
        prior_network=prior_networks["causalpath"],
        weight_type=WeightType.UNIFORM,
    )

    sc.pp.scale(adata)
    dc.mt.decouple(
        adata,
        network,
        methods=comparison_methods,
        tmin=min_targets,
        bsize=decoupler_batch_size,
        verbose=True,
    )

    rows = []
    for method in pending_methods:
        scores = adata.obsm[f"score_{method}"].astype(float).sort_index(axis=1)
        scores.to_parquet(output_paths[method], engine="pyarrow")
        rows.append(
            {
                "dataset": dataset_name,
                "method": method,
                "prior": "causalpath",
                "weight": "UNIFORM",
                "cells": scores.shape[0],
                "regulators": scores.shape[1],
                "output": str(output_paths[method]),
            }
        )
    return rows


In [8]:
comparison_summary = []

for dataset_path in tqdm(dataset_paths, desc="Datasets: comparison methods"):
    comparison_summary.extend(run_comparison_methods_for_dataset(dataset_path))

pd.DataFrame(comparison_summary)


Datasets: comparison methods:   0%|          | 0/2 [00:00<?, ?it/s]

2026-06-20 21:18:16 | [INFO] Reading expression data from: /Users/kisanthapa/Downloads/z-aggregate/reproduce/Reproduce scRNASeq Results/scRNASeq/PapalexiSatija2021_eccite_RNA.h5ad


Loading and preprocessing PapalexiSatija2021_eccite_RNA


2026-06-20 21:18:18 | [INFO]    Loaded data shape: 20729 cells x 18649 genes
2026-06-20 21:18:18 | [INFO] Starting Preprocessing...
2026-06-20 21:18:18 | [INFO]    Filtered cells (min_genes=1000): Removed 0 cells.
2026-06-20 21:18:19 | [INFO]    Filtered genes (min_cells=10): Removed 915 genes.
2026-06-20 21:18:19 | [INFO]    Mitochondrial filter (<20.0%): Removed 0 cells.
2026-06-20 21:18:20 | [INFO] Preprocessing complete. Final shape: 20729 cells x 17734 genes
2026-06-20 21:18:20 | [INFO] Computing weights using strategy: Uniform
2026-06-20 21:18:20 | [INFO]    Uniform weights: using interaction as weight (no overlap filtering).
2026-06-20 21:18:20 | [INFO]    Weights computed successfully.
/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/functools.py:909: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)
2026-06-20 21:18:22 | [INFO] zscore - Running zscore
2026-06-20 21:18:23 | [INFO] Extracted omics mat with 20729

  0%|          | 0/20729 [00:00<?, ?it/s]

...

Loading and preprocessing TianKampmann2021_CRISPRa


2026-06-20 21:26:32 | [INFO]    Loaded data shape: 21193 cells x 33538 genes
2026-06-20 21:26:32 | [INFO] Starting Preprocessing...
2026-06-20 21:26:32 | [INFO]    Filtered cells (min_genes=1000): Removed 941 cells.
2026-06-20 21:26:33 | [INFO]    Filtered genes (min_cells=10): Removed 13170 genes.
2026-06-20 21:26:33 | [INFO]    Mitochondrial filter (<20.0%): Removed 1097 cells.
2026-06-20 21:26:34 | [INFO] Preprocessing complete. Final shape: 19155 cells x 20368 genes
2026-06-20 21:26:34 | [INFO] Computing weights using strategy: Uniform
2026-06-20 21:26:34 | [INFO]    Uniform weights: using interaction as weight (no overlap filtering).
2026-06-20 21:26:34 | [INFO]    Weights computed successfully.
/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/functools.py:909: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)
2026-06-20 21:26:36 | [INFO] zscore - Running zscore
2026-06-20 21:26:37 | [INFO] Extracted omics mat wit

  0%|          | 0/19155 [00:00<?, ?it/s]

2026-06-20 21:31:45 | [INFO] viper - adjusting p-values by FDR
2026-06-20 21:31:45 | [INFO] viper - done


,dataset,method,prior,weight,cells,regulators,output
0,PapalexiSatija2021_eccite_RNA,viper,causalpath,UNIFORM,20729,424,/Users/kisanthapa/Downloads/z-aggregate/reprod...
1,PapalexiSatija2021_eccite_RNA,ulm,causalpath,UNIFORM,20729,424,/Users/kisanthapa/Downloads/z-aggregate/reprod...
2,PapalexiSatija2021_eccite_RNA,zscore,causalpath,UNIFORM,20729,424,/Users/kisanthapa/Downloads/z-aggregate/reprod...
3,TianKampmann2021_CRISPRa,viper,causalpath,UNIFORM,19155,444,/Users/kisanthapa/Downloads/z-aggregate/reprod...
4,TianKampmann2021_CRISPRa,ulm,causalpath,UNIFORM,19155,444,/Users/kisanthapa/Downloads/z-aggregate/reprod...
5,TianKampmann2021_CRISPRa,zscore,causalpath,UNIFORM,19155,444,/Users/kisanthapa/Downloads/z-aggregate/reprod...


## Next step

After all expected datasets and score files are present, run `mwu_delongs.ipynb` to compute the MWU and top-two DeLong results. Re-running this notebook with `overwrite = False` skips score files that already exist.
